# PGR207 Mid-term — Guided Build (CIFAR-10: Architecture x Augmentation x Optimiser)

**How to use this notebook:** Run each cell yourself, one at a time. Before each code section
there's a short markdown block naming the *concepts* it uses — if something doesn't click,
pause and look it up (search terms are given), then come back and re-run.

This notebook builds toward the full mid-term design (7 configs x 3 seeds), but starts with a
**fast smoke-test scale** (tiny subset of data, 1-2 epochs) so you can see the *whole pipeline*
work end-to-end in minutes. At the end there's a clearly marked switch to flip to full scale.

**Roadmap:**
1. OOP crash course (classes, `self`, inheritance) — the thing you said you're shakiest on
2. Setup: device, reproducibility
3. Data: CIFAR-10, transforms (augmentation), train/val/test split
4. Models: a hand-written CNN class + adapting `torchvision` architectures
5. Training loop: the core forward/backward/optimizer mechanics
6. Metrics: accuracy, macro-F1, confusion matrix, mean ± std over seeds
7. Experiment runner: the 7-configuration OFAT design
8. Smoke test (fast) -> switch to full run (slow, run overnight)


## 1. OOP in 10 minutes

**Terms to know:** `class`, `object` (instance), `__init__` (constructor), `self`,
`method`, `attribute`, `inheritance`, `super()`.

A **class** is a blueprint. An **object** is one specific thing built from that blueprint.
`self` just means "this particular object" inside the class's own code.

Every PyTorch model you write will be a class that inherits from `nn.Module`. If the toy
example below makes sense, you already understand 90% of the OOP you need for this project.

If this is unclear after running it, search: *"python classes and objects for beginners"*
or *"python self explained"*.


In [ ]:
class Dog:
    def __init__(self, name, breed):
        # __init__ runs once, when the object is created.
        # self.name / self.breed are ATTRIBUTES: data stored on this specific object.
        self.name = name
        self.breed = breed

    def bark(self):
        # A METHOD: a function that belongs to the class and can use self.<attribute>
        return f"{self.name} says Woof!"


class Puppy(Dog):
    # Puppy INHERITS everything from Dog, then adds/overrides behaviour.
    def __init__(self, name, breed, age_months):
        super().__init__(name, breed)  # runs Dog's __init__ first
        self.age_months = age_months

    def bark(self):
        # Overriding the parent method
        return f"{self.name} (a tiny {self.breed}) says yip!"


rex = Dog("Rex", "Labrador")     # rex is an OBJECT (instance) of class Dog
buddy = Puppy("Buddy", "Poodle", 3)

print(rex.bark())
print(buddy.bark())
print(isinstance(buddy, Dog))   # True -- Puppy IS-A Dog, that's what inheritance means


**Why this matters for PyTorch:** every model you build will look like:

```python
class MyModel(nn.Module):        # <- inheritance, just like Puppy(Dog)
    def __init__(self):
        super().__init__()       # <- exactly like above
        self.layer1 = nn.Conv2d(...)   # <- attributes, but they're layers

    def forward(self, x):        # <- a method PyTorch calls automatically
        x = self.layer1(x)
        return x
```

`forward` is just a method — PyTorch's `nn.Module` base class handles a lot of machinery
(tracking parameters, gradients, `.to(device)`, etc.) so you don't have to write it yourself.
That's the entire point of inheriting from it.


## 2. Setup: device, reproducibility

**Terms:** *device* (CPU vs GPU vs Apple's MPS backend), *random seed*, *reproducibility*.

On Apple Silicon, PyTorch can use the **MPS** backend (Metal Performance Shaders) to run on
your GPU cores instead of the CPU — noticeably faster for CNNs. We detect it automatically
below with a fallback to CPU.

Run this first — if `device` doesn't print `mps`, don't worry, everything still works, just slower.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")

DEVICE = get_device()
print("Using device:", DEVICE)

def set_seed(seed: int):
    """Make results reproducible: same seed -> same weight init, same shuffling."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.mps.manual_seed(seed) if DEVICE.type == "mps" else None


## 3. Data: CIFAR-10, transforms, splits

**Terms:** `Dataset`, `DataLoader`, `transform`, *train/validation/test split*,
*stratified split*, *data augmentation*, *normalisation*.

- A **Dataset** object knows how to fetch one (image, label) pair.
- A **DataLoader** wraps a Dataset and hands you *batches* (e.g. 128 images at a time),
  optionally shuffled.
- A **transform** is a function applied to every image (resize, convert to tensor, augment).

We define **three augmentation strategies** (this is Factor 2 in the assignment):
`none`, `crop_flip` (the classic baseline), and `strong` (crop+flip+RandAugment+erasing).

**Important rule from the assignment:** augmentation is applied to TRAINING data only.
Validation/test always use the same plain deterministic transform (resize/normalise).

If any of this is unclear, search: *"pytorch Dataset and DataLoader explained"* or
*"pytorch torchvision transforms tutorial"*.


In [ ]:
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

# Deterministic transform used for validation AND test, always.
eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])

AUGMENTATIONS = {
    "none": eval_transform,

    "crop_flip": T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]),

    "strong": T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.RandAugment(num_ops=2, magnitude=9),
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
        T.RandomErasing(p=0.25),
    ]),
}

# Downloads CIFAR-10 the first time you run this (needs internet, ~170MB).
full_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
test_set   = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=eval_transform)

# --- Stratified train/val split, created ONCE and reused for every experiment ---
targets = np.array(full_train.targets)
train_idx, val_idx = train_test_split(
    np.arange(len(targets)),
    test_size=0.10,
    stratify=targets,       # keeps class balance in both splits
    random_state=42,        # fixed, NOT one of your experiment seeds -- the split itself never changes
)
print(f"Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_set)}")


In [ ]:
class TransformedSubset(torch.utils.data.Dataset):
    """Wraps CIFAR10 + a list of indices + a transform.
    This is another small example of writing your own class: __init__ stores state,
    __getitem__ and __len__ are 'special methods' Python/PyTorch call automatically.
    """
    def __init__(self, base_dataset, indices, transform):
        self.base = base_dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, label = self.base[self.indices[i]]   # PIL image, int label
        img = self.transform(img)
        return img, label


def make_loaders(aug_name, batch_size=128, subset_size=None):
    """Build train/val/test DataLoaders for a given augmentation strategy.
    subset_size: if set, use only this many TRAINING images (for fast smoke-testing).
    """
    train_transform = AUGMENTATIONS[aug_name]

    t_idx = train_idx if subset_size is None else train_idx[:subset_size]

    train_ds = TransformedSubset(full_train, t_idx, train_transform)
    val_ds   = TransformedSubset(full_train, val_idx, eval_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_set, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


## 4. Models: a hand-written CNN + adapted torchvision architectures

**Terms:** `nn.Conv2d`, `nn.ReLU`, `nn.MaxPool2d`, `nn.Linear`, *feature map*, *flatten*,
*parameter count*.

We need at least 3 architectures (Factor 1). Here: a small hand-written CNN (so you fully
understand one from the inside), plus ResNet-18 and DenseNet-121 from `torchvision`
**trained from scratch** (`weights=None` — required by the assignment) and **adapted** for
32x32 inputs (their default first layers assume 224x224 ImageNet images).


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Each Conv2d: (in_channels, out_channels, kernel_size, padding)
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 32x32 -> 16x16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 16x16 -> 8x8
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 8x8   -> 4x4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
import torchvision.models as models

def build_resnet18_cifar(num_classes=10):
    m = models.resnet18(weights=None, num_classes=num_classes)
    # Adapt for 32x32: default first conv (7x7, stride 2) + maxpool shrink 224->56 way too
    # aggressively for a 32x32 image. Use a smaller conv and drop the maxpool.
    m.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    return m

def build_densenet121_cifar(num_classes=10):
    m = models.densenet121(weights=None, num_classes=num_classes)
    m.features.conv0 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.features.pool0 = nn.Identity()
    return m

ARCHITECTURES = {
    "simple_cnn": SimpleCNN,
    "resnet18":   build_resnet18_cifar,
    "densenet121": build_densenet121_cifar,
}

# Sanity check + parameter counts (report these in your results table)
for name, builder in ARCHITECTURES.items():
    m = builder()
    print(f"{name}: {count_params(m):,} trainable params")


## 5. The training loop: forward, loss, backward, optimizer

**Terms:** *forward pass*, *loss function* (cross-entropy here), *backward pass*
(`loss.backward()`), *gradient*, `optimizer.step()`, `optimizer.zero_grad()`,
*epoch*, *early stopping*, *learning-rate schedule*.

This is the mechanical heart of deep learning. Every model, every framework, does this same
four-step dance per batch: **predict -> measure error -> compute gradients -> update weights.**


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()  # tells layers like BatchNorm/Dropout to behave in "training mode"
    total_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()          # clear old gradients
        outputs = model(images)        # forward pass
        loss = criterion(outputs, labels)
        loss.backward()                # backward pass: computes gradients
        optimizer.step()               # update weights using those gradients

        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()  # no gradients needed for evaluation -> faster, less memory
def evaluate(model, loader, criterion, device):
    model.eval()  # "evaluation mode" (disables dropout, freezes BatchNorm stats)
    total_loss = 0.0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    return total_loss / len(loader.dataset), all_preds, all_labels


In [ ]:
class EarlyStopper:
    """Another small OOP example: this object just REMEMBERS state (best loss seen,
    how many epochs since it improved) between calls to .step()."""
    def __init__(self, patience=5):
        self.patience = patience
        self.best_loss = float("inf")
        self.counter = 0
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
            return False  # not stopping
        else:
            self.counter += 1
            return self.counter >= self.patience  # True -> stop training

    def restore_best(self, model):
        model.load_state_dict(self.best_state)


In [ ]:
OPTIMIZERS = {
    "sgd":   lambda params, lr: optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4),
    "adam":  lambda params, lr: optim.Adam(params, lr=lr, weight_decay=5e-4),
    "adamw": lambda params, lr: optim.AdamW(params, lr=lr, weight_decay=5e-4),
}

# A per-optimizer default learning rate (the assignment says: state your choice and why).
# SGD tolerates/needs a larger LR than Adam-family optimizers -- using the same LR for all
# three would unfairly cripple SGD, so we use a small per-optimizer default here.
DEFAULT_LR = {"sgd": 0.05, "adam": 0.001, "adamw": 0.001}


def train_model(config, epochs, subset_size=None, patience=5, verbose=True):
    """
    config: dict with keys 'architecture', 'augmentation', 'optimizer', 'seed'
    Runs one full training run for one configuration + one seed.
    """
    set_seed(config["seed"])

    train_loader, val_loader, test_loader = make_loaders(config["augmentation"], subset_size=subset_size)

    model = ARCHITECTURES[config["architecture"]]().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    lr = DEFAULT_LR[config["optimizer"]]
    optimizer = OPTIMIZERS[config["optimizer"]](model.parameters(), lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    stopper = EarlyStopper(patience=patience)
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss, _, _ = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        if verbose:
            print(f"  epoch {epoch+1}/{epochs}  train_loss={train_loss:.3f}  val_loss={val_loss:.3f}")

        if stopper.step(val_loss, model):
            if verbose:
                print("  early stopping triggered")
            break

    stopper.restore_best(model)  # use the best-validation-loss weights, not the last epoch
    return model, history, test_loader


## 6. Metrics: accuracy, macro-F1, confusion matrix

**Terms:** *accuracy*, *macro-averaged F1*, *confusion matrix*, *mean ± standard deviation*.

You compute these once per run (per config, per seed), then average across the 3 seeds at the
end — never pool predictions from different seeds together before computing the metric
(the assignment explicitly warns against this: it hides the run-to-run variability).


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

def compute_metrics(preds, labels):
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    cm = confusion_matrix(labels, preds)
    return {"accuracy": acc, "macro_f1": macro_f1, "confusion_matrix": cm}


## 7. The experiment runner: 7 configurations (OFAT design)

**Terms:** *one-factor-at-a-time (OFAT)*, *baseline configuration*, *factor*, *level*.

C1 is the baseline. C2/C3 change only architecture. C4/C5 change only augmentation.
C6/C7 change only optimizer. This mirrors the worked example in your assignment brief.


In [ ]:
BASELINE = {"architecture": "resnet18", "augmentation": "crop_flip", "optimizer": "sgd"}

CONFIGS = {
    "C1": BASELINE,
    "C2": {**BASELINE, "architecture": "simple_cnn"},
    "C3": {**BASELINE, "architecture": "densenet121"},
    "C4": {**BASELINE, "augmentation": "none"},
    "C5": {**BASELINE, "augmentation": "strong"},
    "C6": {**BASELINE, "optimizer": "adam"},
    "C7": {**BASELINE, "optimizer": "adamw"},
}

SEEDS = [0, 1, 2]

for cid, cfg in CONFIGS.items():
    print(cid, cfg)


## 8. SMOKE TEST (fast) — verify the whole pipeline works

This runs **all 7 configs, 1 seed, 2 epochs, on a 2,000-image subset** — just to prove every
piece (data -> model -> training -> metrics) works together, in a few minutes on Apple
Silicon. **This is NOT your real experiment** — it's too small/short to mean anything.
Its only job is to catch bugs cheaply before you commit hours of compute.


In [ ]:
import pandas as pd
import time

results = []

for cid, cfg in CONFIGS.items():
    run_cfg = {**cfg, "seed": 0}
    print(f"--- Smoke test: {cid} {run_cfg} ---")
    t0 = time.time()
    model, history, test_loader = train_model(run_cfg, epochs=2, subset_size=2000, patience=2)
    criterion = nn.CrossEntropyLoss()
    test_loss, preds, labels = evaluate(model, test_loader, criterion, DEVICE)
    metrics = compute_metrics(preds, labels)
    elapsed = time.time() - t0
    print(f"  test_acc={metrics['accuracy']:.3f}  macro_f1={metrics['macro_f1']:.3f}  ({elapsed:.1f}s)")

    results.append({
        "config_id": cid, **cfg, "seed": 0,
        "test_accuracy": metrics["accuracy"],
        "test_macro_f1": metrics["macro_f1"],
        "seconds": elapsed,
    })

smoke_df = pd.DataFrame(results)
smoke_df


## 9. Switching to the REAL experiment (run this later, e.g. overnight)

Once the smoke test above runs without errors, flip these settings and let it run for real.
This is the part that takes genuine compute time — no shortcuts here, this is 21 real
training runs (7 configs x 3 seeds).

- `subset_size=None` -> uses the full ~45,000-image training set
- `epochs=30` (or whatever fixed budget you decide and justify in your notes) -> a full budget
- loops over all 3 `SEEDS`, not just seed 0

**Tip:** Run this as a script in the background (or split across a few sessions) rather than
inside one long-running notebook cell — that way a crash halfway through doesn't lose
everything. Save `all_results.csv` after every run, not just at the end.


In [ ]:
FULL_RUN = False  # <-- flip to True when you're ready to commit real time

if FULL_RUN:
    all_results = []
    for cid, cfg in CONFIGS.items():
        for seed in SEEDS:
            run_cfg = {**cfg, "seed": seed}
            print(f"=== {cid} seed={seed} {run_cfg} ===")
            model, history, test_loader = train_model(run_cfg, epochs=30, subset_size=None, patience=5)
            criterion = nn.CrossEntropyLoss()
            test_loss, preds, labels = evaluate(model, test_loader, criterion, DEVICE)
            metrics = compute_metrics(preds, labels)

            all_results.append({
                "config_id": cid, **cfg, "seed": seed,
                "test_accuracy": metrics["accuracy"],
                "test_macro_f1": metrics["macro_f1"],
            })

            # Save after every single run -- don't lose progress on a crash.
            pd.DataFrame(all_results).to_csv("all_results.csv", index=False)

    full_df = pd.DataFrame(all_results)
    full_df
else:
    print("FULL_RUN is False -- flip it to True when ready. Nothing executed.")


## 10. Aggregating: mean ± std across seeds

Once `all_results.csv` exists with 21 rows, this is how you turn it into the table shape your
assignment wants (mean ± std per configuration, across the 3 seeds).


In [ ]:
# Run this after the full experiment has produced all_results.csv
# full_df = pd.read_csv("all_results.csv")
# summary = full_df.groupby("config_id").agg(
#     accuracy_mean=("test_accuracy", "mean"),
#     accuracy_std=("test_accuracy", "std"),
#     f1_mean=("test_macro_f1", "mean"),
#     f1_std=("test_macro_f1", "std"),
# ).reset_index()
# summary


## Glossary — for your notes

| Term | One-line meaning |
|---|---|
| `nn.Module` | Base class every PyTorch model inherits from |
| `forward()` | Method defining how input flows through the model |
| Epoch | One full pass through the training data |
| Loss function | Measures how wrong the model's predictions are |
| Optimizer | Algorithm that updates weights using gradients (SGD, Adam, AdamW) |
| Learning rate | Step size of each weight update |
| LR schedule | Rule for changing the learning rate over training (e.g. cosine annealing) |
| Early stopping | Stop training when validation loss stops improving |
| Data augmentation | Random transforms applied to training images only |
| OFAT design | Change exactly one factor at a time from a fixed baseline |
| Macro-F1 | F1 score averaged equally across classes (not weighted by frequency) |
| Confusion matrix | Table showing which classes get mistaken for which |
| Seed | Number controlling all randomness, for reproducibility |

**If you want deeper background on any single concept**, good search terms:
*"pytorch nn.Module explained"*, *"cross entropy loss explained"*, *"SGD vs Adam optimizer"*,
*"confusion matrix macro f1 explained"*, *"cosine annealing learning rate"*.
